# Прогноз количества запросов на следующий час

Подготовка почасового ряда, признаков

Модели: `baseline`, `linear`, `rf`, `cat`


In [1]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["axes.grid"] = True


## Настройки


In [2]:
DATA_PATH = "df_with_cat.csv"
city_ids = [34, 16, 35, 6, 7, 42, 1, 38, 21, 66, 41, 27]
CITY_ID = city_ids[0]

TEST_SIZE = 0.20
VALID_SIZE = 0.20
RANDOM_STATE = 42

RF_TREES = 500
CAT_ITERATIONS = 3000


## Подготовка данных


In [3]:
from sklearn.preprocessing import OneHotEncoder

events = pd.read_csv(DATA_PATH)

In [4]:
events["model_response_timestamp"] = pd.to_datetime(
    events["model_response_timestamp"],
    unit="s",
)
events["date_hour"] = events["model_response_timestamp"].dt.floor("h")

events["category"] = events["category"].fillna("unknown").astype(str)

hourly = (
    events.groupby(["number", "category", "date_hour"])
    .size()
    .reset_index(name="count")
)

city_hourly = hourly.loc[
    hourly["number"] == CITY_ID,
    ["date_hour", "category", "count"],
]

all_hours = pd.date_range(
    city_hourly["date_hour"].min(),
    city_hourly["date_hour"].max(),
    freq="h",
)

all_categories = sorted(city_hourly["category"].unique())

full_index = pd.MultiIndex.from_product(
    [all_hours, all_categories],
    names=["date_hour", "category"],
)

base_df = (
    city_hourly.set_index(["date_hour", "category"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

base_df = base_df.sort_values(["date_hour", "category"]).reset_index(drop=True)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

category_ohe = encoder.fit_transform(base_df[["category"]])

category_ohe_df = pd.DataFrame(
    category_ohe,
    columns=encoder.get_feature_names_out(["category"]),
    index=base_df.index,
)

base_df = pd.concat([base_df, category_ohe_df], axis=1)

base_df.head()

,date_hour,category,count,category_Agriculture,"category_Automotive, motorcycles and vehicles","category_Beauty, personal care and wellness","category_Construction, renovation and interior","category_Culture, arts and entertainment",category_Education and training,category_Finance and professional services,...,"category_Manufacturing, industry and equipment","category_Nature, parks and outdoor places",category_Other and unspecified,category_Real estate and business properties,category_Religion and community organizations,category_Retail and trade,category_Sports and active recreation,"category_Tourism, lodging and travel",category_Transportation and logistics,"category_Utilities, security and maintenance services"
0,2026-02-13 21:00:00,Agriculture,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-02-13 21:00:00,"Automotive, motorcycles and vehicles",4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-02-13 21:00:00,"Beauty, personal care and wellness",1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-02-13 21:00:00,"Construction, renovation and interior",0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-02-13 21:00:00,"Culture, arts and entertainment",2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
base_df["count"].sum()

np.int64(17238)

In [6]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(["category", "date_hour"]).reset_index(drop=True).copy()

    result["hour"] = result["date_hour"].dt.hour
    result["day"] = result["date_hour"].dt.day
    result["month"] = result["date_hour"].dt.month
    result["day_of_week"] = result["date_hour"].dt.dayofweek
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)

    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)

    for day in range(7):
        result[f"day_of_week_{day}"] = (result["day_of_week"] == day).astype(int)

    return result


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(["category", "date_hour"]).reset_index(drop=True).copy()

    group = result.groupby("category")["count"]

    result["lag_1"] = group.shift(1)
    result["lag_2"] = group.shift(2)
    result["lag_24"] = group.shift(24)

    result["rolling_mean_3"] = group.transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
    result["rolling_mean_6"] = group.transform(
        lambda x: x.shift(1).rolling(6).mean()
    )
    result["rolling_mean_24"] = group.transform(
        lambda x: x.shift(1).rolling(24).mean()
    )

    return result


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    return add_lag_features(add_time_features(df))

In [7]:
base_df = base_df.sort_values(["category", "date_hour"]).reset_index(drop=True)

featured_df = build_features(base_df)

featured_df["predict_1h"] = featured_df["count"]

excluded_dates = featured_df["month"].eq(2) & featured_df["day"].isin([25])
featured_df = featured_df.loc[~excluded_dates].reset_index(drop=True)
featured_df = featured_df.sort_values(["date_hour"]).reset_index(drop=True)
day_columns = [f"day_of_week_{day}" for day in range(7)]
category_columns = [
    col for col in featured_df.columns
    if col.startswith("category_")
]
feature_cols = [
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_24",
    *day_columns,
    *category_columns,
]
model_df = featured_df.dropna(subset=feature_cols + ["predict_1h"]).reset_index(drop=True)
model_df[feature_cols].head()

,hour_sin,hour_cos,is_weekend,lag_1,lag_2,lag_24,rolling_mean_3,rolling_mean_6,rolling_mean_24,day_of_week_0,...,"category_Manufacturing, industry and equipment","category_Nature, parks and outdoor places",category_Other and unspecified,category_Real estate and business properties,category_Religion and community organizations,category_Retail and trade,category_Sports and active recreation,"category_Tourism, lodging and travel",category_Transportation and logistics,"category_Utilities, security and maintenance services"
0,-0.707107,0.707107,1,3.0,2.0,1.0,2.666667,3.666667,3.166667,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.707107,0.707107,1,1.0,3.0,1.0,2.666667,2.000000,1.791667,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.707107,0.707107,1,0.0,5.0,0.0,1.666667,0.833333,0.291667,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.707107,0.707107,1,0.0,0.0,0.0,0.333333,0.166667,0.208333,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.707107,0.707107,1,15.0,7.0,19.0,10.666667,8.166667,8.250000,0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
model_df["predict_1h"].sum()

np.int64(15980)

## Разделение на train / validation / test


In [9]:
split_index = int(len(model_df) * (1 - TEST_SIZE))
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

valid_index = int(len(train_df) * (1 - VALID_SIZE))
cat_train_df = train_df.iloc[:valid_index].copy()
cat_valid_df = train_df.iloc[valid_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["predict_1h"]

X_cat_train = cat_train_df[feature_cols]
y_cat_train = cat_train_df["predict_1h"]
X_cat_valid = cat_valid_df[feature_cols]
y_cat_valid = cat_valid_df["predict_1h"]

X_test = test_df[feature_cols]
y_test = test_df["predict_1h"]

print(f"train: {X_train.shape}")
print(f"train: {y_train.shape}")
print(f"cat validation: {X_cat_valid.shape}")
print(f"test: {X_test.shape}")


train: (12672, 38)
train: (12672,)
cat validation: (2535, 38)
test: (3168, 38)


## Метрики


In [10]:
scores = []
preds_df = pd.DataFrame({
    "date_hour": test_df["date_hour"].values,
    "category": test_df["category"].values,
    "true": y_test.values,
})


def save_prediction(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    preds_df[name] = y_pred

    scores.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    })


def print_score(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)

    print(f'''
        "model": {name},
        "MAE": {mean_absolute_error(y_true, y_pred)},
        "RMSE": {np.sqrt(mean_squared_error(y_true, y_pred))},
        "R2": {r2_score(y_true, y_pred)},
        "MAPE": {mean_absolute_percentage_error(y_true, y_pred)},
    ''')


## Baseline

Наивный прогноз: считаем, что в следующий час будет столько же запросов, сколько в текущий.


In [11]:
y_pred_baseline = X_test["lag_1"].values
save_prediction("baseline", y_test, y_pred_baseline)
print_score("baseline", y_test, y_pred_baseline)



        "model": baseline,
        "MAE": 0.8939393939393939,
        "RMSE": 1.6196005867515795,
        "R2": -0.14522367356239219,
        "MAPE": 1231097625411253.0,
    


## Linear Regression


In [12]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)
y_pred_linear_train = linear_model.predict(X_train)
print_score("linear", y_test, y_pred_linear)
save_prediction("linear", y_test, y_pred_linear)



        "model": linear,
        "MAE": 0.791619117813881,
        "RMSE": 1.2300157761742507,
        "R2": 0.33946500773670685,
        "MAPE": 1493589193741653.2,
    


## Random Forest


In [13]:
rf_model = RandomForestRegressor(
    n_estimators=RF_TREES,
    max_depth=10,
    min_samples_split=15,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, f"rf_category_model_{CITY_ID}.joblib")

y_pred_rf = rf_model.predict(X_test)
print_score("rf", y_test, y_pred_rf)
save_prediction("rf", y_test, y_pred_rf)



        "model": rf,
        "MAE": 0.7863669355803201,
        "RMSE": 1.188096932072023,
        "R2": 0.38371979976973036,
        "MAPE": 1500664657383528.8,
    


## CatBoost


In [14]:
cat_model = CatBoostRegressor(
    iterations=CAT_ITERATIONS,
    learning_rate=0.01,
    depth=8,
    l2_leaf_reg=3,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=100,
    verbose=200,
)
cat_model.fit(
    X_cat_train,
    y_cat_train,
    eval_set=(X_cat_valid, y_cat_valid),
    use_best_model=True,
)

y_pred_cat = cat_model.predict(X_test)
save_prediction("cat", y_test, y_pred_cat)


0:	learn: 1.7792407	test: 1.7905786	best: 1.7905786 (0)	total: 165ms	remaining: 8m 14s
200:	learn: 1.2758595	test: 1.3969700	best: 1.3969700 (200)	total: 1.29s	remaining: 18s
400:	learn: 1.2161694	test: 1.3777976	best: 1.3777976 (400)	total: 2.4s	remaining: 15.6s
600:	learn: 1.1898934	test: 1.3738769	best: 1.3738659 (599)	total: 3.45s	remaining: 13.8s
800:	learn: 1.1676490	test: 1.3721793	best: 1.3721498 (798)	total: 4.45s	remaining: 12.2s
1000:	learn: 1.1444820	test: 1.3707996	best: 1.3707909 (999)	total: 5.47s	remaining: 10.9s
1200:	learn: 1.1210330	test: 1.3688953	best: 1.3688550 (1189)	total: 6.46s	remaining: 9.68s
1400:	learn: 1.0936023	test: 1.3679947	best: 1.3678054 (1388)	total: 7.46s	remaining: 8.51s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.367610185
bestIteration = 1440

Shrink model to first 1441 iterations.


## Сводная таблица


In [15]:
scores_df = pd.DataFrame(scores).sort_values("RMSE").reset_index(drop=True)
display(scores_df)


,model,MAE,RMSE,R2,MAPE
0,rf,0.786367,1.188097,0.383720,1.500665e+15
1,cat,0.775331,1.191730,0.379945,1.434816e+15
2,linear,0.791619,1.230016,0.339465,1.493589e+15
3,baseline,0.893939,1.619601,-0.145224,1.231098e+15


In [16]:
scores_df = pd.DataFrame(scores).sort_values("RMSE").reset_index(drop=True)
display(scores_df)

model_cols = [
    col for col in preds_df.columns
    if col not in ["date_hour", "category", "true"]
]

category_scores = []

for category, category_df in preds_df.groupby("category"):
    category_true_sum = category_df["true"].sum()
    category_true_share = category_true_sum / preds_df["true"].sum()

    for model in ['rf']:
        y_test_category = category_df["true"].values
        y_pred_category = category_df[model].values

        category_scores.append({
            "category": category,
            "model": model,
            "MAE": mean_absolute_error(y_test_category, y_pred_category),
            "RMSE": np.sqrt(mean_squared_error(y_test_category, y_pred_category)),
            "R2": r2_score(y_test_category, y_pred_category) if len(category_df) > 1 else np.nan,
            "MAPE": mean_absolute_percentage_error(y_test_category, y_pred_category),
        })

category_scores_df = (
    pd.DataFrame(category_scores)
    .sort_values(["category", "RMSE"])
    .reset_index(drop=True)
)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
display(category_scores_df)
print(category_scores_df["MAE"].mean())

,model,MAE,RMSE,R2,MAPE
0,rf,0.786367,1.188097,0.383720,1.500665e+15
1,cat,0.775331,1.191730,0.379945,1.434816e+15
2,linear,0.791619,1.230016,0.339465,1.493589e+15
3,baseline,0.893939,1.619601,-0.145224,1.231098e+15


,category,model,MAE,RMSE,R2,MAPE
0,Agriculture,rf,0.584541,0.700527,0.067111,1.471894e+15
1,"Automotive, motorcycles and vehicles",rf,0.805341,1.079737,0.063721,1.809925e+15
2,"Beauty, personal care and wellness",rf,0.504519,0.611089,-0.039107,1.374827e+15
3,"Construction, renovation and interior",rf,0.489558,0.648669,-0.018456,1.408189e+15
4,"Culture, arts and entertainment",rf,0.596010,0.807726,0.078371,1.325609e+15
5,Education and training,rf,0.596765,0.984805,-0.004727,1.574597e+15
6,Finance and professional services,rf,0.211784,0.297535,-0.221357,7.099515e+14
7,Food and beverages,rf,1.857650,2.570723,0.282499,1.936557e+15
8,Government and municipal services,rf,1.045043,1.319265,0.103817,1.725392e+15
9,Healthcare and medical services,rf,0.879806,1.292586,-0.001106,2.015642e+15


0.78636693558032
